### Imports

In [2]:
import numpy as np
import pandas as pd
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Primary device : {device}')
if torch.cuda.is_available():
    print(f'GPU count      : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}        : {torch.cuda.get_device_name(i)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Primary device : cuda
GPU count      : 2
  GPU 0        : Tesla T4
  GPU 1        : Tesla T4


### Loading and Inspecting the dataset

In [25]:
DATA_PATH = '/kaggle/input/datasets/abdullah23f3061/customer-support-tickets-csv/customer_support_tickets.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [27]:
df.hist

<bound method hist_frame of       Ticket ID        Customer Name              Customer Email  \
0             1        Marisa Obrien  carrollallison@example.com   
1             2         Jessica Rios    clarkeashley@example.com   
2             3  Christopher Robbins   gonzalestracy@example.com   
3             4     Christina Dillon    bradleyolson@example.org   
4             5    Alexander Carroll     bradleymark@example.com   
...         ...                  ...                         ...   
8464       8465           David Todd          adam28@example.net   
8465       8466           Lori Davis       russell68@example.com   
8466       8467      Michelle Kelley        ashley83@example.org   
8467       8468     Steven Rodriguez         fpowell@example.org   
8468       8469      Steven Davis MD          lori20@example.net   

      Customer Age Customer Gender       Product Purchased Date of Purchase  \
0               32           Other              GoPro Hero       2021-03-22 

In [28]:
df.isna().sum()

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

### Doing Data cleaning

In [29]:
FOCUS = ['Ticket Description', 'Ticket Subject', 'Ticket Priority', 'Ticket Type', 'Ticket Channel']

df = df[FOCUS].copy()
df.dropna(subset=['Ticket Description', 'Ticket Priority', 'Ticket Channel'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [30]:
df.head()

,Ticket Description,Ticket Subject,Ticket Priority,Ticket Type,Ticket Channel
0,I'm having an issue with the {product_purchase...,Product setup,Critical,Technical issue,Social media
1,I'm having an issue with the {product_purchase...,Peripheral compatibility,Critical,Technical issue,Chat
2,I'm facing a problem with my {product_purchase...,Network problem,Low,Technical issue,Social media
3,I'm having an issue with the {product_purchase...,Account access,Low,Billing inquiry,Social media
4,I'm having an issue with the {product_purchase...,Data loss,Low,Billing inquiry,Email


In [33]:

print(df.shape)
print('\npriority value counts:')
print(df['Ticket Priority'].value_counts())
print('\nchannel value counts:')
print(df['Ticket Channel'].value_counts())

(8469, 5)

priority value counts:
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

channel value counts:
Ticket Channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64
